In [ ]:
import speech_recognition as sr
import pyttsx3
import pandas as pd
from sklearn.tree import DecisionTreeClassifier

# -------------------------
# TEXT TO SPEECH (Offline)
# -------------------------
engine = pyttsx3.init()

def speak(text):
    print("Assistant:", text)
    engine.say(text)
    engine.runAndWait()

# -------------------------
# SPEECH TO TEXT
# -------------------------
def listen():
    r = sr.Recognizer()
    try:
        with sr.Microphone() as source:
            speak("Tell your symptoms")
            audio = r.listen(source)
        text = r.recognize_google(audio)
        print("You:", text)
        return text.lower()
    except (sr.UnknownValueError, sr.RequestError, AttributeError, ModuleNotFoundError):
        speak("Sorry, I could not understand the audio or could not access the microphone.")
        speak("Please type your symptoms instead:")
        text = input("You (type symptoms): ")
        return text.lower()
    except Exception as e:
        speak(f"An unexpected error occurred: {e}")
        speak("Please type your symptoms instead:")
        text = input("You (type symptoms): ")
        return text.lower()

# -------------------------
# TRAIN SIMPLE ML MODEL
# -------------------------

data = {
    "fever":       [1,1,0,0,1,0],
    "cough":       [1,1,1,0,0,0],
    "headache":    [1,0,1,1,0,0],
    "fatigue":     [1,1,1,1,0,1],
    "body_pain":   [1,0,1,0,1,1],
    "disease": ["Flu","Cold","Migraine","Migraine","Food Poisoning","Food Poisoning"]
}

df = pd.DataFrame(data)

X = df.drop("disease", axis=1)
y = df["disease"]

model = DecisionTreeClassifier()
model.fit(X, y)

# -------------------------
# MEDICINE SUGGESTIONS
# -------------------------

medicine = {
    "Flu": "Paracetamol, rest, drink warm fluids",
    "Cold": "Antihistamine, steam inhalation",
    "Migraine": "Ibuprofen, rest in dark room",
    "Food Poisoning": "ORS, hydration, light food"
}

# -------------------------
# SYMPTOM PROCESSING
# -------------------------

def predict(symptoms_text):

    symptoms = {
        "fever": 1 if "fever" in symptoms_text else 0,
        "cough": 1 if "cough" in symptoms_text else 0,
        "headache": 1 if "headache" in symptoms_text else 0,
        "fatigue": 1 if "fatigue" in symptoms_text else 0,
        "body_pain": 1 if "pain" in symptoms_text else 0,
    }

    input_data = pd.DataFrame([symptoms])
    disease = model.predict(input_data)[0]

    return disease

# -------------------------
# MAIN PROGRAM
# -------------------------

speak("Offline AI Healthcare Assistant Started")

user_text = listen()

if user_text:
    disease = predict(user_text)

    speak(f"You may have {disease}")

    advice = medicine.get(disease, "Consult a doctor")

    speak("Suggested care:")
    speak(advice)

    speak("If symptoms are severe, please visit a hospital.")

Assistant: Offline AI Healthcare Assistant Started
Could not import the PyAudio C module '_portaudio'.
Assistant: Sorry, I could not understand the audio or could not access the microphone.
Assistant: Please type your symptoms instead:
You (type symptoms): cold
Assistant: You may have Cold
Assistant: Suggested care:
Assistant: Antihistamine, steam inhalation
Assistant: If symptoms are severe, please visit a hospital.
